# Farm X: descriptive analysis of herd and milk (PFOI, Component 1)

Descriptive only. This notebook does **not** use or change the forecasting model.

1. **Herd count over time**, split into milking cows, dry cows, heifers, calves and bulls,
   plus deaths (and other exits) per month.
2. **Total herd milk yield over time**, full history, by source.

**Privacy rules (AGENTS.md):** every query aggregates in SQL over the herd-level table
`model_input.day_panel`, so the notebook never receives animal keys or animal-level rows.
Sessions are read-only. Connection settings come from the root `.env` through
`pfoi.config.DatabaseSettings`; nothing is printed.
**Clear all outputs before committing** (Kernel → Restart & Clear Output), and do not share
the charts outside the authorised team.

Set `USE_SYNTHETIC = True` to run the notebook on generated fake data (no database).

In [ ]:
USE_SYNTHETIC = False   # True = fictional demo data, no database connection

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# make `pfoi` importable when the notebook is opened from notebooks/
COMPONENT_DIR = Path.cwd().resolve()
if COMPONENT_DIR.name == "notebooks":
    COMPONENT_DIR = COMPONENT_DIR.parent
sys.path.insert(0, str(COMPONENT_DIR))

pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
plt.rcParams.update({"figure.figsize": (13, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

# Herd segments shown in the charts (day_panel stock columns)
SEGMENTS = {
    "Milking cows": ["milking"],
    "Dry cows": ["dry_open", "dry_pregnant"],
    "Heifers": ["heifers_open", "heifers_pregnant"],
    "Calves": ["calves"],
    "Bulls": ["bulls"],
    "Unknown state": ["unknown_state"],
}
COLORS = {"Milking cows": "#2a6fdb", "Dry cows": "#e08a1e", "Heifers": "#3aa66a",
          "Calves": "#b55fc4", "Bulls": "#7a5230", "Unknown state": "#a0a0a0"}

## 1. Load monthly and daily aggregates

* **Herd stocks**: monthly *average* head count per day (`AVG` over days in the month), so
  a month's value is the typical herd size that month.
* **Events**: monthly *totals* (deaths, culls, sales, births, calvings, entries).
* **Milk**: herd litres per day (one value per day for the whole herd) and monthly sums.
  `target_litres` is the official series: milk shipped before 2020-08-01, DelPro
  (parlour) total from 2020-08-01, empty in the data gap.

In [ ]:
HERD_MONTH_SQL = """
SELECT date_trunc('month', date)::date AS month,
       COUNT(*)                         AS days,
       AVG(n_herd)::float               AS n_herd,
       AVG(milking)::float              AS milking,
       AVG(dry_open)::float             AS dry_open,
       AVG(dry_pregnant)::float         AS dry_pregnant,
       AVG(heifers_open)::float         AS heifers_open,
       AVG(heifers_pregnant)::float     AS heifers_pregnant,
       AVG(calves)::float               AS calves,
       AVG(bulls)::float                AS bulls,
       AVG(unknown_state)::float        AS unknown_state,
       SUM(deaths)::float               AS deaths,
       SUM(culls)::float                AS culls,
       SUM(sales)::float                AS sales,
       SUM(exits)::float                AS exits,
       SUM(births)::float               AS births,
       SUM(calvings)::float             AS calvings,
       SUM(external_entries)::float     AS external_entries
FROM {mi}.day_panel
GROUP BY 1 ORDER BY 1
"""

MILK_DAY_SQL = """
SELECT date,
       target_source,
       target_litres::float     AS target_litres,
       milk_shipped::float      AS milk_shipped,
       milk_delpro_total::float AS milk_delpro_total,
       milking::float           AS milking,
       COALESCE(data_gap, false)       AS data_gap,
       COALESCE(is_suspect_low, false) AS is_suspect_low
FROM {mi}.day_panel
ORDER BY date
"""


def synthetic_panel() -> pd.DataFrame:
    """FICTIONAL demo data with the same columns as day_panel (labelled as such)."""
    rng = np.random.default_rng(7)
    d = pd.date_range("2015-08-01", "2026-08-31", freq="D")
    t = np.arange(len(d))
    season = np.sin(2 * np.pi * d.dayofyear / 365.25)
    f = pd.DataFrame({"date": d})
    f["milking"] = (120 + 25 * np.sin(t / 900) + 6 * season).round()
    f["dry_open"] = (8 + 2 * season).round()
    f["dry_pregnant"] = (22 + 4 * np.cos(t / 400)).round()
    f["heifers_open"] = (40 + 8 * np.sin(t / 700)).round()
    f["heifers_pregnant"] = (25 + 5 * np.cos(t / 600)).round()
    f["calves"] = (55 + 10 * season).round()
    f["bulls"] = (6 + 2 * np.sin(t / 1500)).round()
    f["unknown_state"] = np.where(d < "2017-01-01", 4, 0)
    stocks = ["milking", "dry_open", "dry_pregnant", "heifers_open", "heifers_pregnant",
              "calves", "bulls", "unknown_state"]
    f["n_herd"] = f[stocks].sum(axis=1)
    for c, lam in {"deaths": 0.03, "culls": 0.04, "sales": 0.03, "births": 0.15,
                   "calvings": 0.15, "external_entries": 0.01}.items():
        f[c] = rng.poisson(lam, len(f))
    f["exits"] = f[["deaths", "culls", "sales"]].sum(axis=1)
    per_cow = 11 + 1.5 * season + rng.normal(0, 0.6, len(f))
    litres = f["milking"] * per_cow
    early = d < "2020-08-01"
    f["milk_shipped"] = np.where(early, litres * 0.95, litres * 0.9)
    f["milk_delpro_total"] = np.where(early, np.nan, litres)
    gap = (d >= "2022-03-01") & (d < "2022-06-01")
    f.loc[gap, "milk_delpro_total"] = np.nan
    f["data_gap"] = gap
    f["target_source"] = np.where(early, "shipment", np.where(gap, None, "delpro"))
    f["target_litres"] = np.where(early, f["milk_shipped"], f["milk_delpro_total"])
    f["is_suspect_low"] = False
    return f


if USE_SYNTHETIC:
    _p = synthetic_panel()
    _p["month"] = _p["date"].dt.to_period("M").dt.to_timestamp()
    herd_m = _p.groupby("month").agg(
        days=("date", "size"),
        **{c: (c, "mean") for c in ["n_herd", "milking", "dry_open", "dry_pregnant",
                                    "heifers_open", "heifers_pregnant", "calves", "bulls",
                                    "unknown_state"]},
        **{c: (c, "sum") for c in ["deaths", "culls", "sales", "exits", "births",
                                   "calvings", "external_entries"]},
    ).reset_index()
    milk_d = _p[["date", "target_source", "target_litres", "milk_shipped",
                 "milk_delpro_total", "milking", "data_gap", "is_suspect_low"]].copy()
    DATA_LABEL = "SYNTHETIC DEMO DATA (fictional)"
else:
    from sqlalchemy import text
    from pfoi.data import SqlHerdSource          # read-only session, settings from .env

    src = SqlHerdSource.from_env()
    with src.engine.connect() as conn:
        herd_m = pd.read_sql(text(HERD_MONTH_SQL.format(mi=src.mi)), conn)
        milk_d = pd.read_sql(text(MILK_DAY_SQL.format(mi=src.mi)), conn)
    DATA_LABEL = "Farm X (model_input.day_panel)"

herd_m["month"] = pd.to_datetime(herd_m["month"])
milk_d["date"] = pd.to_datetime(milk_d["date"])
for name, frame in {"herd months": herd_m, "milk days": milk_d}.items():
    k = "month" if "month" in frame else "date"
    print(f"{DATA_LABEL} | {name}: {len(frame):,} rows, "
          f"{frame[k].min():%Y-%m-%d} to {frame[k].max():%Y-%m-%d}")

## 2. Herd count over time

### 2.1 Segments
Monthly average head count per day. Dry = dry open + dry pregnant; heifers = open +
pregnant. "Unknown state" is kept so the segments add up to the total herd.

In [ ]:
for seg, cols in SEGMENTS.items():
    herd_m[seg] = herd_m[cols].sum(axis=1)

# check: the segments must add up to n_herd
diff = (herd_m[list(SEGMENTS)].sum(axis=1) - herd_m["n_herd"]).abs()
print(f"Max |segments - total herd| over all months: {diff.max():.3f} head")

latest = herd_m.iloc[-1]
print(f"Latest month {latest['month']:%Y-%m}: total herd {latest['n_herd']:.0f} "
      f"(" + ", ".join(f"{s} {latest[s]:.0f}" for s in SEGMENTS) + ")")

In [ ]:
fig, ax = plt.subplots()
ax.plot(herd_m["month"], herd_m["n_herd"], color="black", lw=2.2, label="Total herd")
for seg in SEGMENTS:
    ax.plot(herd_m["month"], herd_m[seg], lw=1.4, color=COLORS[seg], label=seg)
ax.set_title(f"Herd size by segment (monthly average head count) — {DATA_LABEL}")
ax.set_ylabel("Head (average per day)")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(ncol=4, loc="upper left", frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# Stacked area: composition of the herd
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
segs = list(SEGMENTS)
ax1.stackplot(herd_m["month"], *[herd_m[s] for s in segs],
              labels=segs, colors=[COLORS[s] for s in segs], alpha=0.85)
ax1.set_title("Herd composition (stacked head count)")
ax1.set_ylabel("Head")
ax1.legend(ncol=6, loc="upper left", frameon=False, fontsize=9)

share = herd_m[segs].div(herd_m["n_herd"], axis=0) * 100
ax2.stackplot(herd_m["month"], *[share[s] for s in segs],
              colors=[COLORS[s] for s in segs], alpha=0.85)
ax2.set_title("Herd composition (% of total herd)")
ax2.set_ylabel("% of herd"); ax2.set_ylim(0, 100)
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout(); plt.show()

### 2.2 Deaths and other exits
Deaths are events, not a herd segment, so they are shown as monthly counts. Culls and
sales are added for context, with births and external entries as the inflows.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
w = 25  # bar width in days
ax1.bar(herd_m["month"], herd_m["deaths"], width=w, color="#c0392b", label="Deaths")
ax1.bar(herd_m["month"], herd_m["culls"], width=w, bottom=herd_m["deaths"],
        color="#e67e22", label="Culls")
ax1.bar(herd_m["month"], herd_m["sales"], width=w,
        bottom=herd_m["deaths"] + herd_m["culls"], color="#95a5a6", label="Sales")
ax1.plot(herd_m["month"], herd_m["deaths"].rolling(12, min_periods=6).mean(),
         color="black", lw=1.5, label="Deaths, 12-month rolling mean")
ax1.set_title("Exits per month"); ax1.set_ylabel("Animals")
ax1.legend(ncol=4, frameon=False)

ax2.bar(herd_m["month"], herd_m["births"], width=w, color="#3aa66a", label="Births")
ax2.bar(herd_m["month"], herd_m["external_entries"], width=w, bottom=herd_m["births"],
        color="#2a6fdb", label="External entries")
ax2.set_title("Inflows per month"); ax2.set_ylabel("Animals")
ax2.legend(ncol=2, frameon=False)
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout(); plt.show()

### 2.3 Check: do the segments add up every day, and do exits explain any gap?
Monthly averages can hide daily gaps, so this checks each day inside SQL and returns
**monthly aggregates only**:

* `gap_days`, `gap_sum`: days where `n_herd` differs from the sum of the eight state
  columns, and the summed difference.
* `exit_days`, `exit_days_with_gap`: whether those gaps sit on days with deaths, culls or sales.
* Flow balance: the day-to-day change in `n_herd` should equal births + entries − exits.
  It is tested twice, with exits counted **on the same day** and **on the previous day**,
  because an animal that leaves on day *t* may still be counted in day *t*'s herd.
  Whichever version has fewer mismatches shows how the ETL dates exits.


In [ ]:
HERD_CHECK_SQL = """
WITH d AS (
  SELECT date,
         n_herd - (calves + heifers_open + heifers_pregnant + milking + dry_open
                   + dry_pregnant + bulls + unknown_state)            AS gap,
         exits,
         n_herd - LAG(n_herd) OVER (ORDER BY date)                     AS change,
         births + external_entries - exits                             AS flow_same_day,
         births + external_entries - LAG(exits) OVER (ORDER BY date)   AS flow_prev_day,
         date - LAG(date) OVER (ORDER BY date)                         AS step
  FROM {mi}.day_panel
)
SELECT date_trunc('month', date)::date                                      AS month,
       COUNT(*)                                                             AS days,
       SUM(CASE WHEN gap <> 0 THEN 1 ELSE 0 END)                            AS gap_days,
       SUM(gap)::float                                                      AS gap_sum,
       SUM(CASE WHEN exits > 0 THEN 1 ELSE 0 END)                           AS exit_days,
       SUM(CASE WHEN exits > 0 AND gap <> 0 THEN 1 ELSE 0 END)              AS exit_days_with_gap,
       SUM(CASE WHEN step = 1 THEN 1 ELSE 0 END)                            AS flow_days,
       SUM(CASE WHEN step = 1 AND change <> flow_same_day THEN 1 ELSE 0 END) AS mismatch_same_day,
       SUM(CASE WHEN step = 1 AND change <> flow_prev_day THEN 1 ELSE 0 END) AS mismatch_prev_day,
       SUM(CASE WHEN step = 1 THEN change END)::float                       AS change_sum,
       SUM(CASE WHEN step = 1 THEN flow_same_day END)::float                AS flow_same_day_sum
FROM d
GROUP BY 1 ORDER BY 1
"""


def herd_check_frame(p: pd.DataFrame) -> pd.DataFrame:
    """Same aggregates as HERD_CHECK_SQL, computed in pandas (synthetic data only)."""
    p = p.sort_values("date").copy()
    states = ["calves", "heifers_open", "heifers_pregnant", "milking", "dry_open",
              "dry_pregnant", "bulls", "unknown_state"]
    p["gap"] = p["n_herd"] - p[states].sum(axis=1)
    p["change"] = p["n_herd"].diff()
    p["flow_same_day"] = p["births"] + p["external_entries"] - p["exits"]
    p["flow_prev_day"] = p["births"] + p["external_entries"] - p["exits"].shift()
    p["ok"] = p["date"].diff().dt.days.eq(1)
    p["month"] = p["date"].dt.to_period("M").dt.to_timestamp()
    ok = p["ok"]
    flags = pd.DataFrame({
        "month": p["month"], "days": 1, "gap_days": p["gap"] != 0, "gap_sum": p["gap"],
        "exit_days": p["exits"] > 0, "exit_days_with_gap": (p["exits"] > 0) & (p["gap"] != 0),
        "flow_days": ok,
        "mismatch_same_day": ok & (p["change"] != p["flow_same_day"]),
        "mismatch_prev_day": ok & (p["change"] != p["flow_prev_day"]),
        "change_sum": p["change"].where(ok, 0), "flow_same_day_sum": p["flow_same_day"].where(ok, 0),
    })
    return flags.groupby("month").sum().reset_index()


if USE_SYNTHETIC:
    # NB: the synthetic stocks are not built from the events, so the flow check fails by design
    herd_chk = herd_check_frame(_p)
else:
    with src.engine.connect() as conn:
        herd_chk = pd.read_sql(text(HERD_CHECK_SQL.format(mi=src.mi)), conn)
herd_chk["month"] = pd.to_datetime(herd_chk["month"])

t = herd_chk.sum(numeric_only=True)
print(f"{DATA_LABEL} | herd consistency, all {int(t['days']):,} days")
print(f"  days where total herd != sum of states: {int(t['gap_days']):,} "
      f"(summed gap {t['gap_sum']:,.0f} head-days)")
print(f"  days with exits: {int(t['exit_days']):,}; of those with a gap: {int(t['exit_days_with_gap']):,}")
print(f"  flow balance mismatches over {int(t['flow_days']):,} consecutive days: "
      f"exits same day {int(t['mismatch_same_day']):,}, exits previous day {int(t['mismatch_prev_day']):,}")
print(f"  net change in n_herd {t['change_sum']:,.0f} vs births + entries - exits {t['flow_same_day_sum']:,.0f}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
ax1.bar(herd_chk["month"], herd_chk["gap_days"], width=25, color="#c0392b",
        label="Days where total herd != sum of states")
ax1.bar(herd_chk["month"], herd_chk["exit_days_with_gap"], width=12, color="black",
        label="... of which days with exits")
ax1.set_title("Total herd vs segments, days with a gap per month"); ax1.set_ylabel("Days")
ax1.legend(frameon=False)

ax2.plot(herd_chk["month"], 100 * herd_chk["mismatch_same_day"] / herd_chk["flow_days"].replace(0, np.nan),
         color="#e67e22", lw=1.5, label="Exits counted same day")
ax2.plot(herd_chk["month"], 100 * herd_chk["mismatch_prev_day"] / herd_chk["flow_days"].replace(0, np.nan),
         color="#2a6fdb", lw=1.5, label="Exits counted previous day")
ax2.set_title("Days where change in herd != births + entries - exits (% of days)")
ax2.set_ylabel("% of days"); ax2.legend(frameon=False)
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout(); plt.show()


### 2.3 Yearly summary table (aggregates)

In [ ]:
herd_m["year"] = herd_m["month"].dt.year
herd_y = herd_m.groupby("year").agg(
    months=("month", "size"),
    total_herd_avg=("n_herd", "mean"),
    **{f"{s.lower().replace(' ', '_')}_avg": (s, "mean") for s in SEGMENTS},
    deaths=("deaths", "sum"), culls=("culls", "sum"), sales=("sales", "sum"),
    births=("births", "sum"), calvings=("calvings", "sum"),
    external_entries=("external_entries", "sum"),
)
herd_y["death_rate_pct_of_avg_herd"] = 100 * herd_y["deaths"] / herd_y["total_herd_avg"]
herd_y  # note: first and last years may be partial (see `months`)

## 3. Total herd milk yield over time (full history)

* `target_litres` (official series): **shipment** before 2020-08-01, **DelPro** from
  2020-08-01, missing in the data gap.
* Both raw sources are also plotted where they exist, so the switch and their difference
  are visible.
* Days flagged `is_suspect_low` (below 50% of the past 28-day mean) are marked.

In [ ]:
md_ = milk_d.copy()
md_["month"] = md_["date"].dt.to_period("M").dt.to_timestamp()
milk_m = md_.groupby("month").agg(
    days=("date", "size"),
    days_with_target=("target_litres", "count"),
    target_litres_sum=("target_litres", "sum"),
    target_litres_per_day=("target_litres", "mean"),
    shipped_sum=("milk_shipped", "sum"),
    shipped_days=("milk_shipped", "count"),
    delpro_sum=("milk_delpro_total", "sum"),
    delpro_days=("milk_delpro_total", "count"),
    gap_days=("data_gap", "sum"),
    suspect_low_days=("is_suspect_low", "sum"),
    milking_cow_days=("milking", "sum"),
).reset_index()
# litres per milking cow per day, only on days that have a target value
with_t = md_[md_["target_litres"].notna()]
per_cow = with_t.groupby("month").agg(l=("target_litres", "sum"), c=("milking", "sum"))
milk_m["litres_per_milking_cow_day"] = milk_m["month"].map(per_cow["l"] / per_cow["c"].replace(0, np.nan))

src_days = md_["target_source"].fillna("missing").value_counts()
print("Days by target source:", src_days.to_dict())
print(f"Total litres in target series: {md_['target_litres'].sum():,.0f} "
      f"over {md_['target_litres'].count():,} days")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5.5))
ax.plot(md_["date"], md_["milk_shipped"], lw=0.5, color="#e08a1e", alpha=0.55, label="Shipped (daily)")
ax.plot(md_["date"], md_["milk_delpro_total"], lw=0.5, color="#2a6fdb", alpha=0.55, label="DelPro total (daily)")
roll = md_.set_index("date")["target_litres"].rolling("30D", min_periods=15).mean()
ax.plot(roll.index, roll.values, color="black", lw=1.8, label="Target series, 30-day mean")
sus = md_[md_["is_suspect_low"]]
ax.scatter(sus["date"], sus["target_litres"], s=10, color="red", zorder=3, label="Suspect-low day")

# shade data-gap and missing-target spans
miss = md_["target_litres"].isna().to_numpy()
starts = np.flatnonzero(miss & ~np.r_[False, miss[:-1]])
ends = np.flatnonzero(miss & ~np.r_[miss[1:], False])
for i, (s, e) in enumerate(zip(starts, ends)):
    ax.axvspan(md_["date"].iloc[s], md_["date"].iloc[e], color="grey", alpha=0.15,
               label="No target value" if i == 0 else None)
ax.axvline(pd.Timestamp("2020-08-01"), color="purple", ls="--", lw=1)
ax.text(pd.Timestamp("2020-08-15"), ax.get_ylim()[1] * 0.95, "source switch\nshipment → DelPro",
        color="purple", fontsize=9, va="top")
ax.set_title(f"Herd milk yield per day, full history — {DATA_LABEL}")
ax.set_ylabel("Litres per day (whole herd)")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(ncol=3, frameon=False, loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
full = milk_m["days_with_target"] == milk_m["days"]
ax1.bar(milk_m["month"], milk_m["target_litres_sum"] / 1000, width=25,
        color=np.where(full, "#2a6fdb", "#b0c4de"))
ax1.set_title("Monthly total milk, target series (light bars = month with missing days)")
ax1.set_ylabel("Thousand litres")

ax2.plot(milk_m["month"], milk_m["litres_per_milking_cow_day"], color="#3aa66a", lw=1.6)
ax2.set_title("Litres per milking cow per day (target litres / milking cow-days)")
ax2.set_ylabel("Litres / cow / day")
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout(); plt.show()

In [ ]:
# Seasonality: average daily herd litres by calendar month, per year
md_["year"] = md_["date"].dt.year
season = md_.pivot_table(index=md_["date"].dt.month, columns="year",
                         values="target_litres", aggfunc="mean")
fig, ax = plt.subplots(figsize=(11, 5))
cmap = plt.get_cmap("viridis", season.shape[1])
for i, y in enumerate(season.columns):
    ax.plot(season.index, season[y], marker="o", ms=3, lw=1.2, color=cmap(i), label=str(y))
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_title("Seasonal pattern: mean litres per day by calendar month")
ax.set_ylabel("Litres per day"); ax.legend(ncol=2, fontsize=8, frameon=False, bbox_to_anchor=(1, 1))
plt.tight_layout(); plt.show()

### 3.1 Yearly milk summary table (aggregates)

In [ ]:
milk_y = md_.groupby("year").agg(
    days=("date", "size"),
    days_with_target=("target_litres", "count"),
    total_litres=("target_litres", "sum"),
    mean_litres_per_day=("target_litres", "mean"),
    min_litres_per_day=("target_litres", "min"),
    max_litres_per_day=("target_litres", "max"),
    shipped_total=("milk_shipped", "sum"),
    delpro_total=("milk_delpro_total", "sum"),
    gap_days=("data_gap", "sum"),
    suspect_low_days=("is_suspect_low", "sum"),
)
milk_y["coverage_pct"] = 100 * milk_y["days_with_target"] / milk_y["days"]
_l = with_t.assign(year=with_t["date"].dt.year).groupby("year")
milk_y["litres_per_milking_cow_day"] = _l["target_litres"].sum() / _l["milking"].sum().replace(0, np.nan)
milk_y

## 4. Notes for interpretation

* Herd counts come from presence intervals (`core.herd_day` → `day_panel`), not from
  summing events, because the source entry/exit balance fails in some months.
* Milk before 2020-08 is shipped milk; from 2020-08 it is the parlour (DelPro) total.
  The two are not the same quantity (shipped excludes milk used on the farm), so a level
  change at the switch is partly a definition change, not only a real change.
* Months at the start and end of the history can be partial; check the `days` columns.
* All values are herd-level aggregates. **Clear outputs before committing this notebook.**